# Alexandria Personality Quiz + Recommendation API

**What this notebook does:**
1. Runs a 10-question BFI-10 personality quiz → computes OCEAN scores
2. Saves the OCEAN scores to **Firebase Firestore**, tied to a `user_id`
3. Recommends the single best-matching place in Alexandria using an LLM (Groq)
4. Exposes everything as a **FastAPI** server, tunneled publicly via **ngrok**,
   so the React app can call it directly:

| Method | Endpoint              | Purpose                                              |
|--------|------------------------|-------------------------------------------------------|
| GET    | `/quiz`                | Get the 10 BFI-10 questions (for the frontend to render) |
| POST   | `/quiz/submit`         | Submit answers → computes + saves OCEAN scores → returns recommendation |
| GET    | `/profile/{user_id}`   | Fetch a saved OCEAN profile                          |
| POST   | `/recommend/{user_id}` | Re-run recommendation using the saved profile        |

**Setup required before running:**
- A Groq API key (https://console.groq.com/keys)
- A Firebase project with Firestore enabled, and a **service account JSON key**
  (Firebase Console → Project Settings → Service Accounts → Generate new private key)
- An ngrok account + authtoken (free, https://ngrok.com) to expose the API publicly

Run the cells top to bottom. The last cell starts the server and prints your
public URL — give that URL to the React team.

In [ ]:
!pip install -q groq fastapi uvicorn pyngrok nest-asyncio firebase-admin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.2 MB/s eta 0:00:00


## 1. Configure secrets

**Do not hardcode keys in cells you might share.** Use Colab's built-in Secrets
manager instead:

1. Click the 🔑 (key) icon in the left sidebar of Colab.
2. Add three secrets:
   - `GROQ_API_KEY` → your Groq key
   - `NGROK_AUTH_TOKEN` → your ngrok authtoken
   - `FIREBASE_CREDENTIALS_JSON` → paste the **entire contents** of your Firebase
     service account JSON file as the value (it's just text)
3. Toggle "Notebook access" ON for each secret.

If you're not using Colab Secrets, you'll be prompted to paste them in manually
as a fallback (less secure — avoid sharing the notebook afterward if you do this).

In [ ]:
import os
import json

def get_secret(name, prompt_text=None):
    """Try Colab Secrets first, fall back to manual input."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    # Fallback: manual entry (use getpass so it's not echoed/stored in cell output)
    from getpass import getpass
    return getpass(prompt_text or f"Enter {name}: ")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "Enter your Groq API key: ")
NGROK_AUTH_TOKEN = get_secret("NGROK_AUTH_TOKEN", "Enter your ngrok authtoken: ")
FIREBASE_CREDENTIALS_JSON = get_secret(
    "FIREBASE_CREDENTIALS_JSON",
    "Paste your Firebase service account JSON (entire file contents): "
)

print("✅ Secrets loaded (values are hidden).")

✅ Secrets loaded (values are hidden).


In [ ]:
from groq import Groq
import requests

client = Groq(api_key=GROQ_API_KEY)

resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {GROQ_API_KEY}"}
)
all_models = [m["id"] for m in resp.json().get("data", [])]

SKIP = ["whisper", "vision", "tts", "guard", "tool"]
chat_models = [m for m in all_models if not any(s in m.lower() for s in SKIP)]

PREFERRED = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
    "llama3-70b-8192",
    "llama3-8b-8192",
    "mixtral-8x7b-32768",
    "gemma2-9b-it",
]

selected_model = next((m for m in PREFERRED if m in chat_models), None)
if not selected_model and chat_models:
    selected_model = sorted(chat_models)[0]

print(f"✅ Groq client ready. Using model: {selected_model}")
print(f"   ({len(chat_models)} chat models available on this account)")

✅ Groq client ready. Using model: llama-3.3-70b-versatile
   (12 chat models available on this account)


In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore

cred_dict = json.loads(FIREBASE_CREDENTIALS_JSON)
cred = credentials.Certificate(cred_dict)

# Avoid "app already exists" error if this cell is re-run
if not firebase_admin._apps:
    firebase_admin.initialize_app(cred)

db = firestore.client()
print("✅ Firebase Firestore connected.")
print(f"   Project: {cred_dict.get('project_id')}")

✅ Firebase Firestore connected.
   Project: rega-63259


## 2. Database schema

Two top-level Firestore collections are used. Notebook 2 (the game) reads
from `users` to shape its questions, and writes to `game_sessions`.

**`users/{user_id}`**
```
{
  "ocean_scores": {
    "openness": 3.5,
    "conscientiousness": 2.0,
    "extraversion": 4.0,
    "agreeableness": 3.0,
    "neuroticism": 2.5
  },
  "ocean_levels": {
    "openness": "High", "conscientiousness": "Medium", ...
  },
  "raw_answers": {"1": 4, "2": 5, ...},   # original BFI-10 answers, for re-scoring later
  "quiz_history": [                         # appended on every retake, oldest→newest
    {"ocean_scores": {...}, "taken_at": "<ISO timestamp>"}
  ],
  "created_at": "<ISO timestamp>",
  "updated_at": "<ISO timestamp>"
}
```

**`game_sessions/{session_id}`** (written by Notebook 2, schema shown here for reference)
```
{
  "user_id": "...",
  "questions": [...],
  "score": 7,
  "total": 10,
  "created_at": "<ISO timestamp>"
}
```

In [ ]:
from datetime import datetime, timezone

def now_iso():
    return datetime.now(timezone.utc).isoformat()

def save_ocean_profile(user_id: str, ocean_scores: dict, ocean_levels: dict, raw_answers: dict):
    """Saves/updates a user's OCEAN profile, keeping a history of past quiz attempts."""
    user_ref = db.collection("users").document(user_id)
    snapshot = user_ref.get()

    entry = {"ocean_scores": ocean_scores, "taken_at": now_iso()}

    if snapshot.exists:
        existing = snapshot.to_dict()
        history = existing.get("quiz_history", [])
        history.append(entry)
        user_ref.update({
            "ocean_scores": ocean_scores,
            "ocean_levels": ocean_levels,
            "raw_answers": raw_answers,
            "quiz_history": history,
            "updated_at": now_iso(),
        })
    else:
        user_ref.set({
            "ocean_scores": ocean_scores,
            "ocean_levels": ocean_levels,
            "raw_answers": raw_answers,
            "quiz_history": [entry],
            "created_at": now_iso(),
            "updated_at": now_iso(),
        })

    return user_ref.get().to_dict()


def load_ocean_profile(user_id: str):
    """Returns the saved profile dict, or None if the user hasn't taken the quiz yet."""
    snapshot = db.collection("users").document(user_id).get()
    return snapshot.to_dict() if snapshot.exists else None


print("✅ DB helper functions ready: save_ocean_profile(), load_ocean_profile()")

✅ DB helper functions ready: save_ocean_profile(), load_ocean_profile()


In [ ]:
BFI10_QUESTIONS = [
    {"number": 1,  "text": "I see myself as someone who... is reserved",                       "trait": "extraversion",      "reversed": True},
    {"number": 2,  "text": "I see myself as someone who... is generally trusting",              "trait": "agreeableness",     "reversed": False},
    {"number": 3,  "text": "I see myself as someone who... tends to be lazy",                   "trait": "conscientiousness", "reversed": True},
    {"number": 4,  "text": "I see myself as someone who... is relaxed and handles stress well", "trait": "neuroticism",       "reversed": True},
    {"number": 5,  "text": "I see myself as someone who... has few artistic interests",         "trait": "openness",          "reversed": True},
    {"number": 6,  "text": "I see myself as someone who... is outgoing and sociable",           "trait": "extraversion",      "reversed": False},
    {"number": 7,  "text": "I see myself as someone who... tends to find fault with others",    "trait": "agreeableness",     "reversed": True},
    {"number": 8,  "text": "I see myself as someone who... does a thorough job",                "trait": "conscientiousness", "reversed": False},
    {"number": 9,  "text": "I see myself as someone who... gets nervous easily",                "trait": "neuroticism",       "reversed": False},
    {"number": 10, "text": "I see myself as someone who... has an active imagination",          "trait": "openness",          "reversed": False},
]

SCALE_LABELS = {1: "Strongly Disagree", 2: "Disagree", 3: "Neutral", 4: "Agree", 5: "Strongly Agree"}

print(f"✅ {len(BFI10_QUESTIONS)} BFI-10 questions loaded.")

✅ 10 BFI-10 questions loaded.


In [ ]:
def compute_ocean_scores(answers: dict) -> dict:
    """
    answers: {question_number (int or str): rating 1-5}
    Returns OCEAN scores, each averaged from 2 items, range 1.0-5.0.
    """
    answers = {int(k): v for k, v in answers.items()}

    def reverse(score):
        return 6 - score

    return {
        "extraversion":      (reverse(answers[1]) + answers[6]) / 2,
        "agreeableness":     (answers[2] + reverse(answers[7])) / 2,
        "conscientiousness": (reverse(answers[3]) + answers[8]) / 2,
        "neuroticism":       (reverse(answers[4]) + answers[9]) / 2,
        "openness":          (reverse(answers[5]) + answers[10]) / 2,
    }


def interpret_score(score: float) -> str:
    if score >= 3.5:
        return "High"
    elif score >= 2.5:
        return "Medium"
    else:
        return "Low"


def levels_for(scores: dict) -> dict:
    return {trait: interpret_score(val) for trait, val in scores.items()}


print("✅ Scoring functions ready: compute_ocean_scores(), interpret_score(), levels_for()")

✅ Scoring functions ready: compute_ocean_scores(), interpret_score(), levels_for()


In [ ]:
def load_places(json_path="alexandria_places.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    places = data["places"]
    print(f"✅ Loaded {len(places)} places from {json_path}")
    return places

PLACES = load_places("alexandria_places.json")

RECOMMEND_SYSTEM_PROMPT = """
You are an expert Alexandria, Egypt tourism recommendation agent.
Analyze the visitor's OCEAN personality scores and recommend the SINGLE best matching place.

OCEAN Guide:
- Openness HIGH (3.5-5): museums, history, culture, art, hidden gems, unusual experiences
- Openness LOW (1-2.4): popular mainstream spots, familiar malls, well-known restaurants
- Extraversion HIGH (3.5-5): social hubs, lively beaches, nightlife, busy restaurants, malls
- Extraversion LOW (1-2.4): quiet cafes, calm parks, solo attractions, peaceful beaches
- Conscientiousness HIGH (3.5-5): structured tours, landmark museums, guided experiences
- Conscientiousness LOW (1-2.4): spontaneous markets, street food, unplanned exploration
- Neuroticism HIGH (3.5-5): calm, peaceful, low-crowd places ONLY — avoid chaos and crowds
- Neuroticism LOW (1-2.4): stable person — any environment works including crowded ones
- Agreeableness HIGH (3.5-5): family-friendly, community, group dining, cooperative settings
- Agreeableness LOW (1-2.4): independent solo experiences

RULES:
1. Recommend exactly ONE place — the single best match for this personality
2. Give 2-3 sentences explaining WHY it fits this specific personality
3. Give one practical visiting tip
4. Warm, enthusiastic travel-guide tone
5. Return ONLY valid JSON — no text before or after:

{
  "introduction": "2-3 sentence warm intro summarizing their personality",
  "recommendation": {
    "name": "exact place name from database",
    "category": "category",
    "reason": "why this is the perfect match for their OCEAN profile",
    "tip": "one practical visiting tip"
  },
  "closing_tip": "one overall Alexandria travel tip for this personality"
}
"""


def get_recommendations(ocean_scores: dict, places: list, model: str = None) -> dict:
    if model is None:
        model = selected_model

    lines = [
        f"{p['name']} ({p['category']}) | atmosphere: {p['atmosphere']} | "
        f"crowd: {p['crowd_level']} | price: {p['price_range']} | "
        f"ideal for: {p['ideal_for']} | {p['description'][:150]}"
        for p in places
    ]
    compressed = "\n".join(lines)

    user_message = f"""
Visitor OCEAN personality scores:
- Openness:          {ocean_scores['openness']:.1f}/5.0  [{interpret_score(ocean_scores['openness'])}]
- Conscientiousness: {ocean_scores['conscientiousness']:.1f}/5.0  [{interpret_score(ocean_scores['conscientiousness'])}]
- Extraversion:      {ocean_scores['extraversion']:.1f}/5.0  [{interpret_score(ocean_scores['extraversion'])}]
- Agreeableness:     {ocean_scores['agreeableness']:.1f}/5.0  [{interpret_score(ocean_scores['agreeableness'])}]
- Neuroticism:       {ocean_scores['neuroticism']:.1f}/5.0  [{interpret_score(ocean_scores['neuroticism'])}]

Alexandria places database:
{compressed}

Pick the single best place for this personality. Return ONLY the JSON.
"""

    try:
        response = client.chat.completions.create(
            model=model,
            max_tokens=400,
            temperature=0.7,
            messages=[
                {"role": "system", "content": RECOMMEND_SYSTEM_PROMPT},
                {"role": "user", "content": user_message},
            ],
        )
    except Exception as e:
        error_msg = str(e)
        if "rate_limit" in error_msg or "429" in error_msg:
            remaining = [m for m in sorted(chat_models) if m != model]
            if remaining:
                return get_recommendations(ocean_scores, places, model=remaining[0])
        return {"error": error_msg}

    raw = response.choices[0].message.content
    if "```json" in raw:
        raw = raw.split("```json")[1].split("```")[0].strip()
    elif "```" in raw:
        raw = raw.split("```")[1].split("```")[0].strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"raw": raw}


print("✅ Recommendation function ready: get_recommendations()")

✅ Loaded 63 places from alexandria_places.json
✅ Recommendation function ready: get_recommendations()


## 3. FastAPI server

This defines the HTTP API the React app will call. Request/response shapes:

**GET `/quiz`**
```json
{ "questions": [ {"number": 1, "text": "...", "scale": {"1": "Strongly Disagree", ...}}, ... ] }
```

**POST `/quiz/submit`**
```json
// request
{ "user_id": "abc123", "answers": {"1": 4, "2": 5, "3": 2, ..., "10": 3} }

// response
{
  "ocean_scores": {...},
  "ocean_levels": {...},
  "recommendation": { "introduction": "...", "recommendation": {...}, "closing_tip": "..." }
}
```

**GET `/profile/{user_id}`** → returns the saved profile dict, or 404 if not found.

**POST `/recommend/{user_id}`** → re-runs recommendation off the saved profile
(useful if the React app wants a fresh suggestion without retaking the quiz).

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from typing import Dict

app = FastAPI(title="Alexandria Personality & Recommendation API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],   # narrow this to your React app's domain before going to production
    allow_methods=["*"],
    allow_headers=["*"],
)


class QuizSubmission(BaseModel):
    user_id: str
    answers: Dict[str, int]   # {"1": 4, "2": 5, ...}


@app.get("/quiz")
def get_quiz():
    return {
        "questions": [
            {"number": q["number"], "text": q["text"]} for q in BFI10_QUESTIONS
        ],
        "scale": SCALE_LABELS,
    }


@app.post("/quiz/submit")
def submit_quiz(payload: QuizSubmission):
    if len(payload.answers) != 10:
        raise HTTPException(status_code=400, detail="Expected answers for all 10 questions.")

    try:
        ocean_scores = compute_ocean_scores(payload.answers)
    except KeyError as e:
        raise HTTPException(status_code=400, detail=f"Missing answer for question {e}")

    ocean_levels = levels_for(ocean_scores)
    save_ocean_profile(payload.user_id, ocean_scores, ocean_levels, payload.answers)
    recommendation = get_recommendations(ocean_scores, PLACES)

    return {
        "ocean_scores": ocean_scores,
        "ocean_levels": ocean_levels,
        "recommendation": recommendation,
    }


@app.get("/profile/{user_id}")
def get_profile(user_id: str):
    profile = load_ocean_profile(user_id)
    if profile is None:
        raise HTTPException(status_code=404, detail="No profile found for this user_id.")
    return profile


@app.post("/recommend/{user_id}")
def recommend_for_user(user_id: str):
    profile = load_ocean_profile(user_id)
    if profile is None:
        raise HTTPException(status_code=404, detail="No profile found for this user_id. Take the quiz first.")
    recommendation = get_recommendations(profile["ocean_scores"], PLACES)
    return {"recommendation": recommendation}


@app.get("/health")
def health():
    return {"status": "ok", "model": selected_model}


print("✅ FastAPI app defined with routes: /quiz, /quiz/submit, /profile/{user_id}, /recommend/{user_id}, /health")

✅ FastAPI app defined with routes: /quiz, /quiz/submit, /profile/{user_id}, /recommend/{user_id}, /health


## 3.5. Test the API in-notebook (before exposing it with ngrok)

This cell calls your FastAPI `app` directly in-process — **no server, no ngrok,
no React needed yet.** It uses a fake `test_user_001` so it won't interfere
with real user data, and you can safely re-run it any time.

It checks, in order:
1. `GET /quiz` → questions load correctly
2. `POST /quiz/submit` → scores compute, save to Firestore, and a real
   recommendation comes back from Groq
3. `GET /profile/test_user_001` → the saved profile reads back correctly
4. `POST /recommend/test_user_001` → re-recommending off the saved profile works
5. An error case: submitting with a missing answer → should return 400
6. An error case: fetching a profile that doesn't exist → should return 404

If anything fails, the printed error will tell you exactly which step and why
(bad API key, Firestore permissions, JSON parsing issue from the LLM, etc.)
— much faster to debug here than after wiring up ngrok + React.

In [ ]:
!pip install -q httpx

from fastapi.testclient import TestClient
import json as _json

test_client = TestClient(app)
TEST_USER = "test_user_001"   # safe to reuse — won't collide with real users

def _print_step(title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)


# 1. GET /quiz
_print_step("1. GET /quiz")
r = test_client.get("/quiz")
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
quiz = r.json()
assert len(quiz["questions"]) == 10, f"Expected 10 questions, got {len(quiz['questions'])}"
print(f"✅ Got {len(quiz['questions'])} questions. First one: {quiz['questions'][0]}")


# 2. POST /quiz/submit  (uses the REAL Groq call + REAL Firestore write)
_print_step("2. POST /quiz/submit")
sample_answers = {
    "1": 2, "2": 4, "3": 1, "4": 4, "5": 2,
    "6": 5, "7": 2, "8": 5, "9": 2, "10": 5,
}  # a made-up but internally varied answer set, good for testing
r = test_client.post("/quiz/submit", json={"user_id": TEST_USER, "answers": sample_answers})
print(f"Status: {r.status_code}")
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
submit_result = r.json()
print("OCEAN scores:", submit_result["ocean_scores"])
print("OCEAN levels:", submit_result["ocean_levels"])
print("\nRecommendation:")
print(_json.dumps(submit_result["recommendation"], indent=2, ensure_ascii=False))


# 3. GET /profile/{user_id}
_print_step("3. GET /profile/test_user_001")
r = test_client.get(f"/profile/{TEST_USER}")
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
profile = r.json()
print("✅ Profile loaded from Firestore:")
print(_json.dumps(profile, indent=2, ensure_ascii=False))


# 4. POST /recommend/{user_id}  (re-recommend off the saved profile, no quiz retake)
_print_step("4. POST /recommend/test_user_001")
r = test_client.post(f"/recommend/{TEST_USER}")
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
print("✅ Re-recommendation:")
print(_json.dumps(r.json(), indent=2, ensure_ascii=False))


# 5. Error case: missing answers → expect 400
_print_step("5. Error case — incomplete answers (expect 400)")
bad_answers = {str(i): 3 for i in range(1, 10)}  # only 9 of 10
r = test_client.post("/quiz/submit", json={"user_id": TEST_USER, "answers": bad_answers})
print(f"Status: {r.status_code} (expected 400)")
assert r.status_code == 400, f"Expected 400, got {r.status_code}: {r.text}"
print("✅ Correctly rejected incomplete submission:", r.json())


# 6. Error case: unknown user → expect 404
_print_step("6. Error case — unknown user profile (expect 404)")
r = test_client.get("/profile/this_user_does_not_exist")
print(f"Status: {r.status_code} (expected 404)")
assert r.status_code == 404, f"Expected 404, got {r.status_code}: {r.text}"
print("✅ Correctly returned 404:", r.json())


_print_step("ALL TESTS PASSED ✅")
print("Your API logic works end-to-end (quiz → scoring → Firestore → Groq recommendation).")
print("You're safe to move on to the ngrok cell below to expose this to React.")


1. GET /quiz
✅ Got 10 questions. First one: {'number': 1, 'text': 'I see myself as someone who... is reserved'}

2. POST /quiz/submit
Status: 200
OCEAN scores: {'extraversion': 4.5, 'agreeableness': 4.0, 'conscientiousness': 5.0, 'neuroticism': 2.0, 'openness': 4.5}
OCEAN levels: {'extraversion': 'High', 'agreeableness': 'High', 'conscientiousness': 'High', 'neuroticism': 'Low', 'openness': 'High'}

Recommendation:
{
  "introduction": "This visitor has a unique blend of high openness, extraversion, and agreeableness, with a strong inclination towards structured and culturally rich experiences. They are likely to thrive in environments that offer a mix of social interaction, intellectual stimulation, and exposure to art, history, and culture. With their high conscientiousness, they probably appreciate well-organized and guided experiences.",
  "recommendation": {
    "name": "Bibliotheca Alexandrina",
    "category": "attraction",
    "reason": "This place is a perfect match for their 

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import asyncio

nest_asyncio.apply()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000).public_url # Access .public_url to get the string URL
print("=" * 60)
print(f"🌍 PUBLIC API URL: {public_url}")
print("=" * 60)
print("Share this URL with your React team. Example calls:")
print(f"  GET  {public_url}/quiz")
print(f"  POST {public_url}/quiz/submit")
print(f"  GET  {public_url}/profile/<user_id>")
print("\nLeave this cell running to keep the API alive.\n")

# Configure Uvicorn server manually to run within the existing event loop
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

# Run the server directly in the current event loop
# This is necessary because uvicorn.run() creates a new event loop,
# which conflicts with nest_asyncio and Colab's environment.
loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())

🌍 PUBLIC API URL: https://bagpipe-runner-depict.ngrok-free.dev
Share this URL with your React team. Example calls:
  GET  https://bagpipe-runner-depict.ngrok-free.dev/quiz
  POST https://bagpipe-runner-depict.ngrok-free.dev/quiz/submit
  GET  https://bagpipe-runner-depict.ngrok-free.dev/profile/<user_id>

Leave this cell running to keep the API alive.



INFO:     Started server process [5155]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Finished server process [5155]


KeyboardInterrupt: 

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import asyncio

nest_asyncio.apply()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000).public_url # Ensure .public_url is called to get the string
print("=" * 60)
print(f"🌍 PUBLIC API URL: {public_url}")
print("=" * 60)
print("Share this URL with your React team. Example calls:")
print(f"  GET  {public_url}/quiz")
print(f"  POST {public_url}/quiz/submit")
print(f"  GET  {public_url}/profile/<user_id>")
print("\nLeave this cell running to keep the API alive.\n")

# Configure Uvicorn server
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

# Run the server directly in the current event loop
# This is necessary because uvicorn.run() creates a new event loop,
# which conflicts with nest_asyncio and Colab's environment.
loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())

🌍 PUBLIC API URL: https://bagpipe-runner-depict.ngrok-free.dev
Share this URL with your React team. Example calls:
  GET  https://bagpipe-runner-depict.ngrok-free.dev/quiz
  POST https://bagpipe-runner-depict.ngrok-free.dev/quiz/submit
  GET  https://bagpipe-runner-depict.ngrok-free.dev/profile/<user_id>

Leave this cell running to keep the API alive.



INFO:     Started server process [5155]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [5155]


KeyboardInterrupt: 

In [ ]:
# --- Interactive quiz: answer it yourself right here in the notebook ---

from fastapi.testclient import TestClient
test_client = TestClient(app)

def run_interactive_quiz(user_id="my_test_user"):
    print("=" * 60)
    print("       ALEXANDRIA TOURISM PERSONALITY QUIZ")
    print("=" * 60)
    print("\nRate each statement from 1 to 5:")
    print("  1 = Strongly Disagree   2 = Disagree   3 = Neutral")
    print("  4 = Agree               5 = Strongly Agree\n")

    answers = {}
    for q in BFI10_QUESTIONS:
        while True:
            raw = input(f"Q{q['number']}/10 — {q['text']}\n   Your answer (1-5): ").strip()
            if raw in {"1", "2", "3", "4", "5"}:
                answers[str(q["number"])] = int(raw)
                print(f"   ✓ {SCALE_LABELS[int(raw)]}\n")
                break
            print("   ⚠️  Please enter a number between 1 and 5.\n")

    print("─" * 60)
    print("Submitting your answers...\n")

    r = test_client.post("/quiz/submit", json={"user_id": user_id, "answers": answers})
    if r.status_code != 200:
        print(f"❌ Error {r.status_code}: {r.text}")
        return None

    result = r.json()

    print("=" * 60)
    print("        YOUR PERSONALITY PROFILE (OCEAN)")
    print("=" * 60)
    for trait, score in result["ocean_scores"].items():
        level = result["ocean_levels"][trait]
        bar = "█" * int(score * 4) + "░" * (20 - int(score * 4))
        print(f"{trait.upper():<20} {score:.1f}/5.0  [{level}]")
        print(f"  {bar}")

    rec = result["recommendation"]
    print("\n" + "=" * 60)
    print("   🌊  YOUR PERFECT ALEXANDRIA SPOT  🌊")
    print("=" * 60)
    print("\n📍 ABOUT YOU\n" + rec.get("introduction", ""))
    place = rec.get("recommendation", {})
    if place:
        print(f"\n✨ {place.get('name','').upper()}  ({place.get('category','')})")
        print(f"   Why: {place.get('reason','')}")
        print(f"   Tip: {place.get('tip','')}")
    if rec.get("closing_tip"):
        print(f"\n🗺️  {rec['closing_tip']}")

    return result


# Run it:
result = run_interactive_quiz()

       ALEXANDRIA TOURISM PERSONALITY QUIZ

Rate each statement from 1 to 5:
  1 = Strongly Disagree   2 = Disagree   3 = Neutral
  4 = Agree               5 = Strongly Agree

Q1/10 — I see myself as someone who... is reserved
   Your answer (1-5): 2
   ✓ Disagree

Q2/10 — I see myself as someone who... is generally trusting
   Your answer (1-5): 3
   ✓ Neutral

Q3/10 — I see myself as someone who... tends to be lazy
   Your answer (1-5): 4
   ✓ Agree

Q4/10 — I see myself as someone who... is relaxed and handles stress well
   Your answer (1-5): 5
   ✓ Strongly Agree

Q5/10 — I see myself as someone who... has few artistic interests
   Your answer (1-5): 8
   ⚠️  Please enter a number between 1 and 5.

Q5/10 — I see myself as someone who... has few artistic interests
   Your answer (1-5): 3
   ✓ Neutral

Q6/10 — I see myself as someone who... is outgoing and sociable
   Your answer (1-5): 2
   ✓ Disagree

Q7/10 — I see myself as someone who... tends to find fault with others
   Your a